# Advanced Python OOP: Extending Classes — Problems and Fully Worked Solutions

**Scope:** subclass-specific features; inherited method calls and dynamic dispatch; overriding and `super()`; optional versus mandatory hooks; abstract base classes; instance/class attribute lookup and shadowing; class-aware factories; cooperative multiple inheritance; and extensible systems.

**Format:** 21 increasingly difficult, self-contained problems, each with a specification, executable solution, assertions and an explanation. Additional diagnostic demonstrations, deliberate failure examples, a capstone, and review questions are included. **Requirements:** Python 3.10+; standard library only. Run **Kernel → Restart & Run All** from the top. All deliberately failing operations are caught and asserted, so a successful run completes without an uncaught exception.

**Source alignment:** This notebook expands the uploaded *Extending* lesson. Corrections: its original `Person` example implements `sleep()` but omits `study()`; its later sentence saying `sleep()` is absent is a typo. Returning `NotImplemented` is not the same as enforcing an abstract method; for a normal required method, raise `NotImplementedError`, or use `ABC` and `@abstractmethod` to prohibit incomplete instantiation.

**Conventions:** `P01`, `P02`, ... prefix classes by problem so all examples can run sequentially without redefining earlier examples. Use `Decimal` rather than binary floats for example monetary calculations. `assert` statements are educational checks, not a substitute for a production test suite.

## Learning map

| Problems | Focus |
|---|---|
| 01–04 | Extending inherited behavior, dynamic dispatch, optional hooks, abstract contracts |
| 05–07 | `super()`, subclass state, constructor-dispatch hazards |
| 08–13 | Class vs instance attributes, money-safe policies, class factories, slots |
| 14–16 | Cooperative multiple inheritance and abstract properties |
| 17–20 | Template methods, registration, descriptors, structural typing |
| 21 | Integrated capstone with multiple inheritance-free extensibility |

**How to practice:** cover the solution cell, implement the prompt, write your own edge-case tests, then compare to the complete solution. No third-party packages, file I/O or external services are needed.

## Two diagnostic examples from the lesson

**Diagnosis A:** An inherited method runs with the actual object's `self`. When `Person.routine()` asks for `self.study()`, the runtime searches the `Student` instance and its class hierarchy. If that method is missing everywhere, `AttributeError` is raised. `Person` already contains `sleep()` in the original code.

**Diagnosis B:** `NotImplemented` is a singleton sentinel used primarily by binary special methods to defer an operation. It is not an exception. `NotImplementedError` is an exception, but ABCs are preferable when construction of incomplete implementations must be prevented.

In [46]:
class DiagnosticPerson:
    def eat(self):
        return "eat"

    def sleep(self):
        return "sleep"

    def routine(self):
        return [self.eat(), self.study(), self.sleep()]


class DiagnosticStudent(DiagnosticPerson):
    def study(self):
        return "study"


assert DiagnosticStudent().routine() == ["eat", "study", "sleep"]
try:
    DiagnosticPerson().routine()
except AttributeError as exc:
    assert "study" in str(exc)
else:
    raise AssertionError("Missing study() must raise AttributeError")

class DiagnosticSentinel:
    def routine(self):
        return NotImplemented  # Returns a value; does not enforce an abstract contract.

assert DiagnosticSentinel().routine() is NotImplemented
print("Diagnostics passed: method lookup and abstract-method semantics.")

Diagnostics passed: method lookup and abstract-method semantics.


## Problem 01 — Extend an inherited template without overriding it

**Challenge.** Design a `Person.routine()` that assembles eating, optional activities and sleeping. `Student` adds studying, and `WorkingStudent` adds a shift. Do **not** duplicate `routine()` in subclasses. Return a list of separate activities so tests do not depend on string concatenation.

**Solution approach.** A concrete, documented hook with a harmless default allows intentional extension. Calls to `self.activities()` resolve dynamically at runtime. An inherited method does not bind `self` permanently to its defining class.

In [47]:
class P01Person:
    def eat(self) -> str:
        return "eat"

    def activities(self) -> list[str]:
        """Extension hook: people without special activities return an empty list."""
        return []

    def sleep(self) -> str:
        return "sleep"

    def routine(self) -> list[str]:
        return [self.eat(), *self.activities(), self.sleep()]


class P01Student(P01Person):
    def study(self) -> str:
        return "study"

    def activities(self) -> list[str]:
        return [self.study()]


class P01WorkingStudent(P01Student):
    def work(self) -> str:
        return "work"

    def activities(self) -> list[str]:
        return [*super().activities(), self.work()]

In [48]:
assert P01Person().routine() == ["eat", "sleep"]
assert P01Student().routine() == ["eat", "study", "sleep"]
assert P01WorkingStudent().routine() == ["eat", "study", "work", "sleep"]
assert isinstance(P01WorkingStudent(), P01Person)
assert "routine" not in P01WorkingStudent.__dict__
print("P01 passed: inherited template and chained subclass extensions.")

P01 passed: inherited template and chained subclass extensions.


**Why it works:** `routine()` is defined exactly once; `self.activities()` calls the most specific implementation. `super().activities()` preserves the parent contribution instead of replacing it. Returning a fresh list on each call avoids shared mutable state.

## Problem 02 — Debug a missing subclass-only method

**Challenge.** Reproduce why calling an inherited `run()` on a base instance fails when it blindly calls `self.execute()`. Then refactor it so instantiating an incomplete implementation fails immediately. Prove both failure behaviors using tests.

**Solution approach.** The original lookup is legal but fails only at call time. If `execute()` is mandatory, use `ABC` and `@abstractmethod` so incomplete concrete classes cannot be instantiated.

In [49]:
from abc import ABC, abstractmethod

class P02FragileRunner:
    def run(self):
        return self.execute()


class P02FragileJob(P02FragileRunner):
    def execute(self):
        return "done"


class P02Runner(ABC):
    def run(self) -> str:
        return self.execute()

    @abstractmethod
    def execute(self) -> str:
        """Return one execution result."""


class P02Job(P02Runner):
    def execute(self) -> str:
        return "done"


class P02Incomplete(P02Runner):
    pass

In [50]:
assert P02FragileJob().run() == "done"
try:
    P02FragileRunner().run()
except AttributeError as exc:
    assert "execute" in str(exc)
else:
    raise AssertionError("Expected call-time failure")

try:
    P02Incomplete()
except TypeError as exc:
    assert "execute" in str(exc)
else:
    raise AssertionError("ABC must prevent incomplete instantiation")
assert P02Job().run() == "done"
print("P02 passed: call-time lookup failure vs construction-time ABC enforcement.")

P02 passed: call-time lookup failure vs construction-time ABC enforcement.


**Design decision:** a base class is not automatically abstract merely because its intended subclasses add a method. Declare a requirement with `@abstractmethod`; reserve an optional default hook (Problem 1) for genuinely optional behavior.

## Problem 03 — Why `hasattr` is not a sufficient extension contract

**Challenge.** Investigate a `study` name that exists but is not callable, and a property that raises `AttributeError`. Replace probing with a stable `optional_study()` hook. The base instance must always work, and subclasses must be able to add one activity.

**Solution approach.** `hasattr(obj, name)` invokes normal attribute access and suppresses `AttributeError`; it does not prove a value is callable. A concrete hook makes the class contract explicit.

In [51]:
class P03Unreliable:
    study = "not a method"

    def run(self):
        if hasattr(self, "study"):
            return self.study()
        return "skip"


class P03PropertyTrap:
    @property
    def study(self):
        raise AttributeError("property body failed")


class P03Person:
    def optional_study(self) -> list[str]:
        return []

    def routine(self) -> list[str]:
        return ["eat", *self.optional_study(), "sleep"]


class P03Student(P03Person):
    def optional_study(self) -> list[str]:
        return ["study"]

In [52]:
assert hasattr(P03Unreliable(), "study")
try:
    P03Unreliable().run()
except TypeError:
    pass
else:
    raise AssertionError("Presence alone does not imply callability")

assert not hasattr(P03PropertyTrap(), "study")  # AttributeError was swallowed.
assert P03Person().routine() == ["eat", "sleep"]
assert P03Student().routine() == ["eat", "study", "sleep"]
print("P03 passed: optional hook avoids fragile hasattr probing.")

P03 passed: optional hook avoids fragile hasattr probing.


**If working with truly dynamic plugin objects:** use `candidate = getattr(obj, "study", None)` and `callable(candidate)`, and specify how errors in property getters are handled. Prefer explicit interfaces for normal class designs; probes may trigger descriptors with side effects.

## Problem 04 — Build an extensible abstract routine

**Challenge.** Require every concrete person to supply a `primary_activity()`. Keep the scheduling algorithm in the base class and let a tutor extend a student activity using `super()`. Preserve a custom `name` and reject an empty name.

**Solution approach.** A template method can be concrete while one of its hooks is abstract. `super()` enables an additive override without copying parent logic.

In [53]:
class P04Person(ABC):
    def __init__(self, name: str):
        if not name.strip():
            raise ValueError("name must not be empty")
        self.name = name.strip()

    @abstractmethod
    def primary_activity(self) -> list[str]:
        pass

    def routine(self) -> list[str]:
        return [f"{self.name}: eat", *self.primary_activity(), "sleep"]


class P04Student(P04Person):
    def primary_activity(self) -> list[str]:
        return ["study"]


class P04Tutor(P04Student):
    def primary_activity(self) -> list[str]:
        return [*super().primary_activity(), "tutor"]

In [54]:
assert P04Student(" Alex ").routine() == ["Alex: eat", "study", "sleep"]
assert P04Tutor("Mina").routine() == ["Mina: eat", "study", "tutor", "sleep"]
for make in (lambda: P04Person("X"), lambda: P04Student("  ")):
    try:
        make()
    except (TypeError, ValueError):
        pass
    else:
        raise AssertionError("Abstract/validation contract should reject construction")
print("P04 passed: abstract extension point and inherited scheduling.")

P04 passed: abstract extension point and inherited scheduling.


**Check your mental model:** `P04Tutor` inherits `routine()` from `P04Person`, but the abstract hook is implemented and resolved using the tutor object. An abstract class may contain fully implemented methods and constructors.

## Problem 05 — Stack `super()` overrides without losing behavior

**Challenge.** Implement a document renderer. The base class returns the body; a signed subclass appends a signature; a watermarked signed subclass prefixes a watermark and retains the signature. No subclass may hard-code a parent class method call.

**Solution approach.** Each override transforms `super().render()`. `super()` refers to the next class in the method resolution order (MRO), making chaining possible.

In [55]:
class P05Document:
    def __init__(self, body: str):
        self.body = body

    def render(self) -> str:
        return self.body


class P05SignedDocument(P05Document):
    def __init__(self, body: str, signer: str):
        super().__init__(body)
        self.signer = signer

    def render(self) -> str:
        return f"{super().render()}\nSigned: {self.signer}"


class P05WatermarkedSignedDocument(P05SignedDocument):
    def __init__(self, body: str, signer: str, watermark: str):
        super().__init__(body, signer)
        self.watermark = watermark

    def render(self) -> str:
        return f"[{self.watermark}]\n{super().render()}"

In [56]:
doc = P05WatermarkedSignedDocument("Terms", "Ada", "DRAFT")
assert doc.render() == "[DRAFT]\nTerms\nSigned: Ada"
assert P05SignedDocument("Report", "Bo").render() == "Report\nSigned: Bo"
assert P05Document("Plain").render() == "Plain"
assert P05WatermarkedSignedDocument.__mro__[:3] == (
    P05WatermarkedSignedDocument, P05SignedDocument, P05Document
)
print("P05 passed: three layers of rendering in the intended order.")

P05 passed: three layers of rendering in the intended order.


**Pitfall:** calling `P05Document.render(self)` directly would skip a later class's override in more complex hierarchies. `super()` is an MRO operation, not an unconditional synonym for “my lexical parent.”

## Problem 06 — Extend initialization while preserving invariants

**Challenge.** Create a counter with nonnegative initial value. Extend it into a tagged counter that adds a validated label without duplicating the original initialization or increment logic. Demonstrate that the subtype remains substitutable as a counter.

**Solution approach.** Call `super().__init__` to establish base invariants, then initialize only the new subclass state. Reuse inherited methods unless customization is necessary.

In [57]:
class P06Counter:
    def __init__(self, initial: int = 0):
        if type(initial) is not int or initial < 0:
            raise ValueError("initial must be a nonnegative integer")
        self.value = initial

    def increment(self, step: int = 1) -> int:
        if type(step) is not int or step <= 0:
            raise ValueError("step must be a positive integer")
        self.value += step
        return self.value


class P06TaggedCounter(P06Counter):
    def __init__(self, tag: str, initial: int = 0):
        super().__init__(initial)
        if not tag.strip():
            raise ValueError("tag must not be empty")
        self.tag = tag.strip()

    def describe(self) -> str:
        return f"{self.tag}={self.value}"

In [58]:
counter = P06TaggedCounter(" jobs ", 2)
assert isinstance(counter, P06Counter)
assert counter.increment(3) == 5
assert counter.describe() == "jobs=5"
for constructor in (lambda: P06Counter(-1), lambda: P06TaggedCounter("  ")):
    try:
        constructor()
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid construction must fail")
print("P06 passed: base initialization and subclass-only state.")

P06 passed: base initialization and subclass-only state.


**Invariant:** the inherited `increment()` is still the single implementation of counter arithmetic. Note that validating with `type(x) is int` intentionally rejects `True`, which is an `int` subclass in Python.

## Problem 07 — Find the virtual-call-in-constructor trap

**Challenge.** Create a failing example in which a base constructor calls a method overridden by a child before the child has initialized its own fields. Then redesign construction so validation happens only after all attributes exist.

**Solution approach.** Python dynamically dispatches calls on `self` even inside `__init__`. Avoid overridable calls from partially initialized base objects; use an explicit finalization step or non-overridable local validation.

In [59]:
class P07FragileBase:
    def __init__(self):
        self.ready = True
        self.validate()  # Dangerous: child state might not exist yet.

    def validate(self):
        return None


class P07FragileChild(P07FragileBase):
    def __init__(self):
        super().__init__()
        self.token = "configured"

    def validate(self):
        if not self.token:
            raise ValueError("token required")


class P07SafeBase:
    def __init__(self):
        self.ready = True  # Do not invoke overridable hooks here.

    def validate(self) -> None:
        pass


class P07SafeChild(P07SafeBase):
    def __init__(self, token: str):
        super().__init__()
        self.token = token
        self.validate()  # Object state is now complete.

    def validate(self) -> None:
        if not self.token:
            raise ValueError("token required")

In [60]:
try:
    P07FragileChild()
except AttributeError as exc:
    assert "token" in str(exc)
else:
    raise AssertionError("Expected access to uninitialized subclass state")

assert P07SafeChild("configured").ready
try:
    P07SafeChild("")
except ValueError:
    pass
else:
    raise AssertionError("Token validation must run after initialization")
print("P07 passed: virtual dispatch during construction understood and fixed.")

P07 passed: virtual dispatch during construction understood and fixed.


**Further thought:** an explicit call to `self.validate()` at the *end* of the final subclass constructor works here; it is not universally safe if further subclasses add more initialization afterward. A separate factory or post-construction validation boundary is preferable in open-ended subclass hierarchies.

## Problem 08 — Predict three attribute-lookup strategies

**Challenge.** Implement an account class with `apr=3.0` and a savings subclass with `apr=5.0`. Compare an inherited method using `P08Account.apr`, `self.apr`, and `type(self).apr` after assigning an instance-level `apr=10.0`. Assert all outcomes.

**Solution approach.** A hard-coded base lookup ignores subclass configuration; ordinary instance access permits shadowing; lookup on the actual class honors subclass configuration but ignores an ordinary instance attribute.

In [61]:
class P08Account:
    apr = 3.0

    def __init__(self, identifier: int):
        self.identifier = identifier

    def fixed_base_rate(self):
        return P08Account.apr

    def potentially_shadowed_rate(self):
        return self.apr

    def runtime_class_rate(self):
        return type(self).apr


class P08Savings(P08Account):
    apr = 5.0

In [62]:
generic = P08Account(1)
savings = P08Savings(2)
assert (generic.fixed_base_rate(), generic.potentially_shadowed_rate(), generic.runtime_class_rate()) == (3.0, 3.0, 3.0)
assert (savings.fixed_base_rate(), savings.potentially_shadowed_rate(), savings.runtime_class_rate()) == (3.0, 5.0, 5.0)
savings.apr = 10.0  # This creates an instance attribute; it does not update the class.
assert savings.__dict__["apr"] == 10.0
assert savings.potentially_shadowed_rate() == 10.0
assert savings.runtime_class_rate() == 5.0
assert P08Savings.apr == 5.0
print("P08 passed: base lookup, instance shadowing and runtime-class lookup.")

P08 passed: base lookup, instance shadowing and runtime-class lookup.


**Precise language:** assigning `obj.apr` here *shadows* an inherited class attribute; assigning `Subclass.apr` redefines the attribute at class level. `type(self).apr` expresses a deliberate class-policy choice, not an across-the-board replacement for `self.apr`.

## Problem 09 — Implement subclass-sensitive interest safely with Decimal

**Challenge.** Calculate simple annual interest on a nonnegative balance. Store `apr` as a subclass-level fractional rate, round interest to two decimal places, and prove that accidental `account.apr` shadowing does not alter a class-wide policy. Reject negative balances.

**Solution approach.** Represent money and rates as `Decimal`, quantize with an explicit rule, and use `type(self).apr` to read class-specific policy. These are illustrative calculations, not financial advice.

In [63]:
from decimal import Decimal, ROUND_HALF_UP, InvalidOperation
from typing import ClassVar

class P09Account:
    apr: ClassVar[Decimal] = Decimal("0.03")

    def __init__(self, identifier: str, balance: Decimal):
        if not isinstance(balance, Decimal):
            raise TypeError("balance must be Decimal")
        if balance < 0:
            raise ValueError("balance cannot be negative")
        self.identifier = identifier
        self.balance = balance

    def annual_interest(self) -> Decimal:
        raw = self.balance * type(self).apr
        return raw.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)


class P09Savings(P09Account):
    apr = Decimal("0.05")


class P09PremiumSavings(P09Savings):
    apr = Decimal("0.075")

In [64]:
standard = P09Account("A", Decimal("100.00"))
savings = P09Savings("B", Decimal("100.00"))
premium = P09PremiumSavings("C", Decimal("100.00"))
assert standard.annual_interest() == Decimal("3.00")
assert savings.annual_interest() == Decimal("5.00")
assert premium.annual_interest() == Decimal("7.50")
savings.apr = Decimal("0.99")  # instance shadow does not change the chosen policy
assert savings.annual_interest() == Decimal("5.00")
assert P09Savings.apr == Decimal("0.05")
try:
    P09Savings("bad", Decimal("-0.01"))
except ValueError:
    pass
else:
    raise AssertionError("Negative balances must be rejected")
print("P09 passed: subclass rates, currency rounding and validation.")

P09 passed: subclass rates, currency rounding and validation.


**Why not `float`?** Decimal literals avoid the binary representation surprises common in decimal money calculations. Rounding is made explicit at the business boundary. This problem models **simple annual** interest, not compounding or daily accrual.

## Problem 10 — Support *intentional* per-instance rate overrides

**Challenge.** Revise the rate design so the default APR comes from the actual class, but each account may optionally have its own validated rate. Provide `set_rate()` and `clear_rate()` and test that one account’s changes do not affect other accounts.

**Solution approach.** Do not rely on accidental shadowing. Give the per-instance override a private attribute and express the selection rule with a property.

In [65]:
class P10Account:
    default_apr: ClassVar[Decimal] = Decimal("0.03")

    def __init__(self, balance: Decimal):
        if not isinstance(balance, Decimal) or balance < 0:
            raise ValueError("balance must be a nonnegative Decimal")
        self.balance = balance
        self._override_apr: Decimal | None = None

    @property
    def effective_apr(self) -> Decimal:
        if self._override_apr is not None:
            return self._override_apr
        return type(self).default_apr

    def set_rate(self, rate: Decimal) -> None:
        if not isinstance(rate, Decimal) or not rate.is_finite():
            raise ValueError("rate must be a finite Decimal")
        if not Decimal("0") <= rate <= Decimal("1"):
            raise ValueError("rate must be between 0 and 1")
        self._override_apr = rate

    def clear_rate(self) -> None:
        self._override_apr = None

    def annual_interest(self) -> Decimal:
        return (self.balance * self.effective_apr).quantize(
            Decimal("0.01"), rounding=ROUND_HALF_UP
        )


class P10Savings(P10Account):
    default_apr = Decimal("0.05")

In [66]:
first = P10Savings(Decimal("200"))
second = P10Savings(Decimal("200"))
assert first.annual_interest() == second.annual_interest() == Decimal("10.00")
first.set_rate(Decimal("0.10"))
assert first.effective_apr == Decimal("0.10")
assert first.annual_interest() == Decimal("20.00")
assert second.annual_interest() == Decimal("10.00")
first.clear_rate()
assert first.effective_apr == Decimal("0.05")
for rate in (Decimal("-0.1"), Decimal("1.1"), Decimal("NaN")):
    try:
        first.set_rate(rate)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Should reject {rate!r}")
print("P10 passed: explicit instance policy without contaminating class defaults.")

P10 passed: explicit instance policy without contaminating class defaults.


**Design contrast:** Problem 9 deliberately ignores per-instance `apr`. Problem 10 deliberately supports an override through `effective_apr`. The correct choice depends on the contract, not on cleverness of lookup syntax.

## Problem 11 — Eliminate shared mutable class-state bugs

**Challenge.** Demonstrate how two instances mistakenly share an inherited mutable `events` list. Correct the design to use per-instance lists while retaining an immutable, class-level `allowed_events` configuration. Show that a subclass can redefine the configuration.

**Solution approach.** Class attributes are shared references. Mutable per-object state belongs in `__init__`; immutable configuration can legitimately live on the class.

In [67]:
class P11BadLog:
    events = []  # Demonstration of a bug: shared mutable list.

    def add(self, event: str) -> None:
        self.events.append(event)


class P11Log:
    allowed_events: ClassVar[frozenset[str]] = frozenset({"start", "stop"})

    def __init__(self):
        self.events: list[str] = []

    def add(self, event: str) -> None:
        if event not in type(self).allowed_events:
            raise ValueError(f"unrecognized event: {event}")
        self.events.append(event)


class P11ExtendedLog(P11Log):
    allowed_events = P11Log.allowed_events | {"pause"}

In [68]:
bad1, bad2 = P11BadLog(), P11BadLog()
bad1.add("unexpectedly shared")
assert bad2.events == ["unexpectedly shared"]
# Cleanup is local to the demonstration; production code should not have this shared list.
P11BadLog.events.clear()

log1, log2 = P11Log(), P11Log()
log1.add("start")
assert log1.events == ["start"] and log2.events == []
assert log1.events is not log2.events
extended = P11ExtendedLog()
extended.add("pause")
try:
    log2.add("pause")
except ValueError:
    pass
else:
    raise AssertionError("Only subclass permits pause")
print("P11 passed: instance state is isolated; class configuration is inherited.")

P11 passed: instance state is isolated; class configuration is inherited.


**Extra subtlety:** `self.events += [...]` on a shared class-level list can mutate the same list before any assignment occurs. Explicit initialization is the simplest correction.

## Problem 12 — Write an inherited factory that returns the right subclass

**Challenge.** Add a `from_record()` parser for strings such as `X-1|125.50`. Calling the factory on a savings subclass must produce a savings object, not a base account. Reject malformed records, blanks and negative amounts.

**Solution approach.** Declare a `@classmethod` and construct with `cls(...)` instead of spelling the base class name. Keep record parsing separate from instance initialization.

In [69]:
class P12Account:
    account_type: ClassVar[str] = "generic"

    def __init__(self, number: str, balance: Decimal):
        if not number.strip():
            raise ValueError("empty account number")
        if not balance.is_finite() or balance < 0:
            raise ValueError("balance must be finite and nonnegative")
        self.number = number.strip()
        self.balance = balance

    @classmethod
    def from_record(cls, record: str):
        parts = record.split("|")
        if len(parts) != 2:
            raise ValueError("expected 'number|balance'")
        number, raw_balance = parts
        try:
            balance = Decimal(raw_balance.strip())
        except InvalidOperation as exc:
            raise ValueError("invalid balance") from exc
        return cls(number, balance)


class P12Savings(P12Account):
    account_type = "savings"

In [70]:
item = P12Savings.from_record(" SV-01 |125.50")
assert type(item) is P12Savings
assert item.number == "SV-01" and item.balance == Decimal("125.50")
assert item.account_type == "savings"
assert type(P12Account.from_record("A|0")) is P12Account
for record in ("missing delimiter", "|10", "A|-1", "A|NaN", "A|not-a-number"):
    try:
        P12Savings.from_record(record)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Expected invalid record: {record!r}")
print("P12 passed: inherited factory is polymorphic and validates inputs.")

P12 passed: inherited factory is polymorphic and validates inputs.


**Production refinement:** catch `decimal.InvalidOperation` directly around numeric parsing, as shown. The subclass constructor must remain compatible with `cls(number, balance)`, or override the factory. Reject non-finite numeric inputs after parsing, not by treating them as parse errors.

## Problem 13 — Use slots to restrict accidental instance shadowing

**Challenge.** Create a slotted account and a slotted subclass with a different class-level APR. Show that assigning `instance.apr` is rejected while the subclass APR remains available. Explain why defining `__slots__` only on the parent is not enough.

**Solution approach.** Use `__slots__ = (...)` in the base **and** `__slots__ = ()` in subclasses that add no instance fields. Normally, an unslotted subclass regains `__dict__`. Slots enforce layout restrictions, not comprehensive immutability or security.

In [71]:
class P13Account:
    __slots__ = ("number",)
    apr: ClassVar[Decimal] = Decimal("0.03")

    def __init__(self, number: str):
        self.number = number

    def class_rate(self):
        return type(self).apr


class P13Savings(P13Account):
    __slots__ = ()
    apr = Decimal("0.05")


class P13UnslottedChild(P13Account):
    pass  # Automatically gets a __dict__ again.

In [72]:
locked = P13Savings("S-1")
assert locked.class_rate() == Decimal("0.05")
assert not hasattr(locked, "__dict__")
try:
    locked.apr = Decimal("0.90")
except AttributeError:
    pass
else:
    raise AssertionError("Slotted instance must reject arbitrary apr assignment")

open_child = P13UnslottedChild("U-1")
open_child.apr = Decimal("0.90")
assert open_child.__dict__["apr"] == Decimal("0.90")
assert open_child.class_rate() == Decimal("0.03")
print("P13 passed: slot inheritance and its limitations.")

P13 passed: slot inheritance and its limitations.


**Caution:** `__slots__` is not a replacement for public API design; a user can still change class attributes and, depending on the hierarchy, subclass instances may reintroduce a dictionary.

## Problem 14 — Compose two mixins using the method resolution order

**Challenge.** Create `PrefixMixin` and `SuffixMixin`, each of which extends `render()` cooperatively. Combine them with a concrete core class. Verify the exact output and inspect the MRO. Then reverse the mixin order and predict the new output.

**Solution approach.** Every mixin delegates with `super().render()` and modifies the result. Python follows the C3 MRO; `super()` means *next in that MRO*, not necessarily the class written directly in the source.

In [73]:
class P14Core:
    def render(self) -> str:
        return "core"


class P14PrefixMixin:
    def render(self) -> str:
        return f"prefix({super().render()})"


class P14SuffixMixin:
    def render(self) -> str:
        return f"{super().render()}:suffix"


class P14Pipeline(P14PrefixMixin, P14SuffixMixin, P14Core):
    pass


class P14ReversePipeline(P14SuffixMixin, P14PrefixMixin, P14Core):
    pass

In [74]:
assert P14Pipeline().render() == "prefix(core:suffix)"
assert P14ReversePipeline().render() == "prefix(core):suffix"
assert P14Pipeline.__mro__[:4] == (
    P14Pipeline, P14PrefixMixin, P14SuffixMixin, P14Core
)
assert P14ReversePipeline.__mro__[:4] == (
    P14ReversePipeline, P14SuffixMixin, P14PrefixMixin, P14Core
)
print("P14 pipeline MRO:", " -> ".join(c.__name__ for c in P14Pipeline.__mro__))
print("P14 passed: cooperative method chaining and order sensitivity.")

P14 pipeline MRO: P14Pipeline -> P14PrefixMixin -> P14SuffixMixin -> P14Core -> object
P14 passed: cooperative method chaining and order sensitivity.


**Reasoning trace:** in `P14Pipeline`, Prefix delegates to Suffix, which delegates to Core. In the reversed hierarchy, Suffix delegates to Prefix, which delegates to Core. Mixins must agree on method signatures and return contracts.

## Problem 15 — Initialize a diamond hierarchy exactly once

**Challenge.** Build two mixins that require distinct keyword-only constructor arguments and share one root class. Use cooperative `super()` and `**kwargs` so the root initializes once. Record the initialization trace to prove ordering and reject unused keyword arguments.

**Solution approach.** Each constructor consumes only its own keyword(s) and forwards the rest. The MRO traverses the diamond once. Root exhausts accepted keywords before calling `object.__init__` without arguments.

In [75]:
class P15Root:
    def __init__(self, *, identifier: str, **kwargs):
        if kwargs:
            raise TypeError(f"unexpected arguments: {sorted(kwargs)}")
        super().__init__()  # object.__init__ receives no unexpected keywords.
        self.identifier = identifier
        self.trace = ["root"]


class P15AuditMixin:
    def __init__(self, *, auditor: str, **kwargs):
        super().__init__(**kwargs)
        self.auditor = auditor
        self.trace.append("audit")


class P15CacheMixin:
    def __init__(self, *, cache_size: int, **kwargs):
        if cache_size < 0:
            raise ValueError("negative cache size")
        super().__init__(**kwargs)
        self.cache_size = cache_size
        self.trace.append("cache")


class P15Service(P15AuditMixin, P15CacheMixin, P15Root):
    def __init__(self, *, identifier: str, auditor: str, cache_size: int):
        super().__init__(identifier=identifier, auditor=auditor, cache_size=cache_size)
        self.trace.append("service")

In [76]:
service = P15Service(identifier="S1", auditor="Ada", cache_size=10)
assert (service.identifier, service.auditor, service.cache_size) == ("S1", "Ada", 10)
assert service.trace == ["root", "cache", "audit", "service"]
assert service.trace.count("root") == 1
try:
    P15Service(identifier="x", auditor="y", cache_size=-1)
except ValueError:
    pass
else:
    raise AssertionError("Invalid cache size must be rejected")
print("P15 passed: cooperative constructors follow C3 MRO exactly once.")

P15 passed: cooperative constructors follow C3 MRO exactly once.


**Tradeoff:** cooperative multiple inheritance needs a deliberate shared signature convention; composition is often simpler if mixins cannot reliably cooperate. In particular, blindly forwarding keywords to `object.__init__()` raises `TypeError`.

## Problem 16 — Enforce an abstract property and abstract delivery hook

**Challenge.** Specify a notifier contract requiring both a `channel` property and a `_deliver(message)` method. Provide shared input validation in `send()`. Implement an email subtype, and assert that a subtype implementing only one requirement is still abstract.

**Solution approach.** Mark the property getter and required operation with `@abstractmethod`. The concrete template can call both. Tests should verify failed construction and the concrete return value.

In [77]:
class P16Notifier(ABC):
    @property
    @abstractmethod
    def channel(self) -> str:
        pass

    @abstractmethod
    def _deliver(self, message: str) -> str:
        pass

    def send(self, message: str) -> str:
        if not message.strip():
            raise ValueError("message must not be blank")
        return f"{self.channel}: {self._deliver(message.strip())}"


class P16Incomplete(P16Notifier):
    @property
    def channel(self) -> str:
        return "incomplete"


class P16Email(P16Notifier):
    @property
    def channel(self) -> str:
        return "email"

    def _deliver(self, message: str) -> str:
        return f"queued<{message}>"

In [78]:
assert P16Email().send(" hello ") == "email: queued<hello>"
for cls in (P16Notifier, P16Incomplete):
    try:
        cls()
    except TypeError:
        pass
    else:
        raise AssertionError(f"{cls.__name__} must remain abstract")
try:
    P16Email().send("   ")
except ValueError:
    pass
else:
    raise AssertionError("Blank message should be invalid")
print("P16 passed: abstract property, abstract method and validated template.")

P16 passed: abstract property, abstract method and validated template.


**Important nuance:** ABC enforcement operates at object construction; it does not validate every return type, method signature, or business invariant. Keep runtime validation and tests for those guarantees.

## Problem 17 — Make a template method provide in-memory rollback

**Challenge.** Create an abstract transaction with a concrete `execute()` algorithm: snapshot balances, validate, apply, return a result, and restore balances if validation or application raises. Implement a transfer subclass using `Decimal`. Test a successful transfer and an insufficient-funds failure.

**Solution approach.** Centralize the algorithm and error-handling path in the base, then let subclasses implement only the transaction-specific hooks. This rollback covers only copied in-memory state, not external side effects.

In [79]:
class P17Transaction(ABC):
    def __init__(self, balances: dict[str, Decimal]):
        self.balances = balances.copy()

    @abstractmethod
    def validate(self) -> None:
        pass

    @abstractmethod
    def apply(self) -> str:
        pass

    def execute(self) -> str:
        before = self.balances.copy()
        try:
            self.validate()
            return self.apply()
        except Exception:
            self.balances = before
            raise


class P17Transfer(P17Transaction):
    def __init__(self, balances: dict[str, Decimal], source: str,
                 destination: str, amount: Decimal):
        super().__init__(balances)
        self.source = source
        self.destination = destination
        self.amount = amount

    def validate(self) -> None:
        if self.source not in self.balances or self.destination not in self.balances:
            raise KeyError("unknown account")
        if self.source == self.destination:
            raise ValueError("source and destination must differ")
        if not isinstance(self.amount, Decimal) or not self.amount.is_finite():
            raise ValueError("amount must be a finite Decimal")
        if self.amount <= 0:
            raise ValueError("transfer amount must be positive")
        if self.balances[self.source] < self.amount:
            raise ValueError("insufficient funds")

    def apply(self) -> str:
        self.balances[self.source] -= self.amount
        self.balances[self.destination] += self.amount
        return f"moved {self.amount} from {self.source} to {self.destination}"

In [80]:
original = {"A": Decimal("20.00"), "B": Decimal("2.00")}
transfer = P17Transfer(original, "A", "B", Decimal("7.50"))
assert transfer.execute() == "moved 7.50 from A to B"
assert transfer.balances == {"A": Decimal("12.50"), "B": Decimal("9.50")}
assert original == {"A": Decimal("20.00"), "B": Decimal("2.00")}

rejected = P17Transfer(original, "A", "B", Decimal("100"))
before = rejected.balances.copy()
try:
    rejected.execute()
except ValueError as exc:
    assert "insufficient" in str(exc)
else:
    raise AssertionError("Insufficient funds must fail")
assert rejected.balances == before

class P17FailingAfterDebit(P17Transfer):
    def apply(self) -> str:
        self.balances[self.source] -= self.amount
        raise RuntimeError("simulated in-memory failure after debit")

partial = P17FailingAfterDebit(original, "A", "B", Decimal("3.00"))
try:
    partial.execute()
except RuntimeError as exc:
    assert "simulated" in str(exc)
else:
    raise AssertionError("Expected a failure after changing state")
assert partial.balances == original  # Proves rollback even after a partial mutation.
print("P17 passed: subclass-defined steps and in-memory rollback.")

P17 passed: subclass-defined steps and in-memory rollback.


**Boundary note:** the snapshot is shallow; this works because balances are immutable `Decimal` values. Nested mutable structures need deeper rollback logic. Never treat this miniature example as a durable database transaction.

## Problem 18 — Register subclasses automatically with `__init_subclass__`

**Challenge.** Implement a registry keyed by an explicit subclass keyword `kind`. Every registered concrete transformer must implement `transform(text)`. Create uppercasing and reversing plugins, a factory for kinds, and duplicate/unknown-key error checks.

**Solution approach.** Registration can happen when subclasses are defined, not every time they are instantiated. `__init_subclass__` receives class-definition keywords; call `super().__init_subclass__(**kwargs)` to cooperate with other bases.

In [81]:
class P18Transformer(ABC):
    registry: ClassVar[dict[str, type]] = {}

    def __init_subclass__(cls, *, kind: str | None = None, **kwargs):
        super().__init_subclass__(**kwargs)
        if kind is not None:
            if not kind:
                raise ValueError("kind cannot be empty")
            if kind in P18Transformer.registry:
                raise ValueError(f"duplicate transformer kind: {kind}")
            P18Transformer.registry[kind] = cls

    @abstractmethod
    def transform(self, text: str) -> str:
        pass

    @classmethod
    def create(cls, kind: str):
        try:
            transformer_type = cls.registry[kind]
        except KeyError as exc:
            raise ValueError(f"unknown transformer: {kind}") from exc
        return transformer_type()


class P18Upper(P18Transformer, kind="upper"):
    def transform(self, text: str) -> str:
        return text.upper()


class P18Reverse(P18Transformer, kind="reverse"):
    def transform(self, text: str) -> str:
        return text[::-1]

In [82]:
assert isinstance(P18Transformer.create("upper"), P18Upper)
assert P18Transformer.create("upper").transform("hello") == "HELLO"
assert P18Transformer.create("reverse").transform("hello") == "olleh"
try:
    P18Transformer.create("missing")
except ValueError:
    pass
else:
    raise AssertionError("Unknown key should fail")
try:
    class P18Duplicate(P18Transformer, kind="upper"):
        def transform(self, text: str) -> str:
            return text
except ValueError as exc:
    assert "duplicate" in str(exc)
else:
    raise AssertionError("Duplicate registrations must fail")
assert set(P18Transformer.registry) == {"upper", "reverse"}
print("P18 passed: registration, factory dispatch and duplicate protection.")

P18 passed: registration, factory dispatch and duplicate protection.


**Limitations:** this minimal registry assumes trusted, no-argument plugins defined in the same process. Production designs may need registration-time checks, explicit dependency injection, removal/reload policies and concurrency handling.

## Problem 19 — Explain why some attributes cannot be shadowed normally

**Challenge.** Contrast a non-data descriptor (implements `__get__` only) with a data descriptor (implements both `__get__` and `__set__`). Write directly to `instance.__dict__` using the same attribute names. Predict which descriptor wins, and test direct assignment.

**Solution approach.** Data descriptors take precedence over ordinary instance dictionaries; a regular non-data descriptor can be shadowed by an instance attribute. A `property` is a data descriptor even without an explicit setter.

In [83]:
class P19NonData:
    def __get__(self, obj, owner=None):
        if obj is None:
            return self
        return "non-data descriptor value"


class P19Data:
    def __get__(self, obj, owner=None):
        if obj is None:
            return self
        return "data descriptor value"

    def __set__(self, obj, value):
        raise AttributeError("read-only descriptor")


class P19Example:
    loose = P19NonData()
    strict = P19Data()

In [84]:
example = P19Example()
assert example.loose == "non-data descriptor value"
assert example.strict == "data descriptor value"
example.__dict__["loose"] = "shadowed instance value"
example.__dict__["strict"] = "attempted shadow"
assert example.loose == "shadowed instance value"
assert example.strict == "data descriptor value"
try:
    example.strict = "direct assignment"
except AttributeError as exc:
    assert "read-only" in str(exc)
else:
    raise AssertionError("Data descriptor must intercept assignment")
print("P19 passed: data-descriptor and instance-dictionary precedence.")

P19 passed: data-descriptor and instance-dictionary precedence.


**Lookup refinement:** the simple “instance attributes beat class attributes” rule from Problem 8 applies to ordinary class attributes; *data descriptors are an important exception*. Attribute lookup can also be customized by `__getattribute__` and `__getattr__`.

## Problem 20 — Compare nominal inheritance with a structural Protocol

**Challenge.** Define a `Studier` protocol and a function that needs only a `study()` method. Show that an unrelated robot can satisfy the function without inheriting a Person base class. Demonstrate why `@runtime_checkable` is not a signature validator.

**Solution approach.** Protocols express structural expectations for type checkers. Runtime-checkable protocols support limited `isinstance()` checks; actual method compatibility still needs type checking and tests.

In [85]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class P20Studier(Protocol):
    def study(self) -> str:
        ...


class P20Student:
    def study(self) -> str:
        return "read notes"


class P20Robot:
    def study(self) -> str:
        return "scan dataset"


class P20BadSignature:
    def study(self, topic: str) -> str:
        return topic


def p20_schedule(worker: P20Studier) -> str:
    return f"today: {worker.study()}"

In [86]:
assert p20_schedule(P20Student()) == "today: read notes"
assert p20_schedule(P20Robot()) == "today: scan dataset"
assert isinstance(P20Robot(), P20Studier)
assert isinstance(P20BadSignature(), P20Studier)  # Only limited runtime presence check!
try:
    p20_schedule(P20BadSignature())
except TypeError:
    pass
else:
    raise AssertionError("Incompatible study signature must fail when called")
assert not issubclass(P20Robot, P01Person)
print("P20 passed: structural substitution and runtime-check limitations.")

P20 passed: structural substitution and runtime-check limitations.


**When to choose which:** ABCs help when construction-time enforcement and shared implementation are needed. Protocols help when unrelated objects can meet the same interface. Both demand clear contracts and actual tests; type annotations are not automatically enforced by Python.

## Problem 21 — CAPSTONE — an extensible account activity engine

**Challenge.** Integrate the lesson in one design: an abstract account has validated `Decimal` balance, an inherited class-aware factory, a class-level rate, an abstract activity hook and a reusable report template. A savings subtype adds interest activity; a student savings subtype extends that activity with study time using `super()`. Check dynamic dispatch, subclass factory identity, instance shadowing, errors and independent activity lists.

**Solution approach.** The shared template belongs in the abstract base; subclasses provide only the varying behavior. Use `cls` in the factory, `type(self)` for class-wide rates and `super()` for additive extension. Avoid mutable class-level results.

In [87]:
class P21Account(ABC):
    apr: ClassVar[Decimal] = Decimal("0.02")
    label: ClassVar[str] = "base"

    def __init__(self, number: str, balance: Decimal):
        if not number.strip():
            raise ValueError("number must be nonempty")
        if not isinstance(balance, Decimal) or not balance.is_finite() or balance < 0:
            raise ValueError("balance must be a finite nonnegative Decimal")
        self.number = number.strip()
        self.balance = balance

    @classmethod
    def from_balance(cls, number: str, amount: str):
        try:
            value = Decimal(amount)
        except InvalidOperation as exc:
            raise ValueError("amount must be numeric") from exc
        return cls(number, value)

    def annual_interest(self) -> Decimal:
        return (self.balance * type(self).apr).quantize(
            Decimal("0.01"), rounding=ROUND_HALF_UP
        )

    @abstractmethod
    def activities(self) -> list[str]:
        pass

    def report(self) -> dict[str, object]:
        return {
            "number": self.number,
            "type": type(self).label,
            "interest": self.annual_interest(),
            "activities": list(self.activities()),
        }


class P21Savings(P21Account):
    apr = Decimal("0.05")
    label = "savings"

    def activities(self) -> list[str]:
        return ["review savings"]


class P21StudentSavings(P21Savings):
    apr = Decimal("0.06")
    label = "student savings"

    def study(self) -> str:
        return "study finance"

    def activities(self) -> list[str]:
        return [*super().activities(), self.study()]

In [88]:
savings = P21Savings.from_balance("S-1", "100.00")
student = P21StudentSavings.from_balance("T-1", "100.00")
assert type(savings) is P21Savings and type(student) is P21StudentSavings
assert savings.report() == {
    "number": "S-1", "type": "savings", "interest": Decimal("5.00"),
    "activities": ["review savings"]
}
assert student.report() == {
    "number": "T-1", "type": "student savings", "interest": Decimal("6.00"),
    "activities": ["review savings", "study finance"]
}
student.apr = Decimal("0.99")
assert student.annual_interest() == Decimal("6.00")  # Class policy prevails.
first_report = student.report()
first_report["activities"].append("local edit")
assert "local edit" not in student.report()["activities"]
for make in (lambda: P21Account("A", Decimal("1")),
             lambda: P21Savings.from_balance("", "1"),
             lambda: P21Savings.from_balance("X", "-1"),
             lambda: P21Savings.from_balance("X", "NaN")):
    try:
        make()
    except (TypeError, ValueError):
        pass
    else:
        raise AssertionError("Expected abstract/validation construction failure")
print("P21 passed: polymorphic factory, ABC, dynamic dispatch, subclass policy and isolation.")

P21 passed: polymorphic factory, ABC, dynamic dispatch, subclass policy and isolation.


### Capstone walkthrough

1. `P21StudentSavings.from_balance(...)` inherits the class method; `cls` is the student savings class, so construction preserves subtype identity.
2. `report()` is inherited unchanged. Its calls to `self.annual_interest()` and `self.activities()` operate on the real student savings instance.
3. `annual_interest()` reads `type(self).apr`, selecting the student subclass’s `0.06` rather than `P21Account.apr` or an accidental instance attribute.
4. Student activities call `super().activities()`, preserving the parent contribution, and then append a subclass-specific contribution.
5. Validation blocks abstract instantiation, missing identifiers, negative balances and NaN. Each call produces fresh lists; editing one report does not alter another.

**Further challenge (solve before looking at earlier problems):** implement an `HonorsStudentSavings` subclass with `apr=Decimal("0.065")` and an additional `"mentor peers"` activity, without rewriting `report()`, `annual_interest()` or the factory.

In [89]:
class P21HonorsStudentSavings(P21StudentSavings):
    apr = Decimal("0.065")
    label = "honors student savings"

    def activities(self) -> list[str]:
        return [*super().activities(), "mentor peers"]


honors = P21HonorsStudentSavings.from_balance("H-1", "100.00")
assert type(honors) is P21HonorsStudentSavings
assert honors.report()["interest"] == Decimal("6.50")
assert honors.report()["activities"] == [
    "review savings", "study finance", "mentor peers"
]
print("Capstone extension passed: fourth level requires only subclass-specific code.")

Capstone extension passed: fourth level requires only subclass-specific code.


## Advanced review: predict before executing

For each statement, write your predicted result and justify it using the actual lookup or MRO rule, then run the verification cell.

1. Can `P01Person()` complete `routine()` without studying? What design choice makes this safe?
2. Does overriding `P08Savings.apr` change `P08Account.apr`? Does assigning `savings.apr` change either class?
3. Why does `P15Service` initialize the root only once even though there are two mixins?
4. Does returning `NotImplemented` prevent a base-class object from being created? Would `NotImplementedError` alone prevent construction?
5. Can a data descriptor be shadowed by writing a key into `instance.__dict__`?
6. Does `isinstance(obj, P20Studier)` guarantee `obj.study()` needs no arguments?
7. Why must `P13Savings` also define `__slots__ = ()`?
8. When is a Protocol preferable to an ABC, and what does each *not* guarantee?

**Answers:** (1) Yes: default hook. (2) Neither assignment changes the base; the instance assignment changes only that instance's lookup. (3) C3 MRO and each constructor calls `super()` once. (4) No; returning a sentinel never enforces construction, and raising in a method fails only when that method is called. (5) No: data descriptors win ordinary lookup. (6) No: runtime checking is limited. (7) Otherwise an unslotted subclass generally acquires a `__dict__`. (8) Protocol for unrelated structural implementers; ABC for shared behavior plus construction-time abstract enforcement. Neither guarantees business correctness or runtime type annotation enforcement.

In [90]:
# Final non-mutating smoke tests across the entire notebook.
assert P01WorkingStudent().routine()[1:3] == ["study", "work"]
assert P02Job().run() == "done"
assert P04Tutor("Mina").routine()[1:3] == ["study", "tutor"]
assert P05Document("body").render() == "body"
assert P09PremiumSavings("x", Decimal("200")).annual_interest() == Decimal("15.00")
assert P10Savings(Decimal("100")).effective_apr == Decimal("0.05")
assert P12Savings.from_record("z|1").account_type == "savings"
assert P14Pipeline().render() == "prefix(core:suffix)"
assert P16Email().send("ok") == "email: queued<ok>"
assert P18Transformer.create("reverse").transform("abc") == "cba"
assert P21HonorsStudentSavings.from_balance("h", "100").annual_interest() == Decimal("6.50")
print("ALL 21 PROBLEMS + CAPSTONE EXTENSION: SMOKE TESTS PASSED")

ALL 21 PROBLEMS + CAPSTONE EXTENSION: SMOKE TESTS PASSED


## Reusable best-practice checklist

- State whether a method is a **required abstract hook** or an **optional hook with a safe default**. Do not confuse these contracts.
- Keep shared algorithms in the base and changing behavior in subclasses; use `super()` for **additive** overrides.
- Never assume `self` refers to the class where a method was written; method and attribute lookup use the actual instance and MRO.
- Do not call overridable methods while an object is partly initialized unless the contract explicitly accounts for it.
- Call `super().__init__` when extending construction; for cooperative multiple inheritance, consume and forward named keywords consistently.
- Choose deliberately between `self.rate` (can see an instance shadow), `type(self).rate` (runtime-class policy) and a base-class-qualified lookup (fixed base policy).
- Put mutable instance state in `__init__`, not on the class. Remember data descriptors and slots affect ordinary lookup and assignment.
- Prefer `Decimal` for exact decimal financial examples; validate domain inputs and define rounding.
- Use inherited `@classmethod` factories with `cls(...)` when constructor contracts are compatible.
- Test normal behavior, invalid inputs, subclass identity, shared-state leaks and dynamic dispatch. Be explicit about in-memory-only versus external guarantees.

**End of notebook.** Run all cells to verify the solutions.